# Multi-Architecture Model Training & Evaluation Pipeline

Trains and evaluates 8 facial attribute classification architectures under strict **StratifiedGroupKFold** cross-validation on `person_id`.
Guarantees **0% identity leakage** between training folds and final holdout test set.


In [ ]:
# Cell 1: Setup environment, paths, and random seed
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Add repository root to Python path
PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name in ['training', 'analysis', 'inference', 'verification'] else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Fix seeds for deterministic execution
np.random.seed(42)
torch.manual_seed(42)
print(f"Project root resolved: {PROJECT_ROOT}")
print(f"PyTorch CUDA available: {torch.cuda.is_available()}")


In [ ]:
# Cell 2: Ingest dataset metadata and partition identities
from sklearn.model_selection import GroupShuffleSplit

from src.models.train_and_evaluate import load_dataset

# Load sample dataset (or increase to full dataset)
df = load_dataset(sample_limit=1000, target_attr="Smiling")
target_attr = "Smiling"
feature_cols = [c for c in df.columns if c not in ["image_id", "person_id", target_attr]]
X = df[feature_cols].values
y = df[target_attr].values
groups = df["person_id"].values

# Partition into identity-disjoint train (85%) and holdout test (15%)
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

# Assert 0.0% identity leakage
overlap = set(groups_train).intersection(set(groups_test))
assert len(overlap) == 0, f"Identity leakage: {len(overlap)} overlapping identities!"
print(f"Train samples: {len(y_train)} across {len(set(groups_train))} identities")
print(f"Test samples: {len(y_test)} across {len(set(groups_test))} identities")
print("Identity leakage check: 0.0% overlap (PASS)")


In [ ]:
# Cell 3: Feature standardization and StratifiedGroupKFold setup
from sklearn.model_selection import StratifiedGroupKFold

# Compute mean and standard deviation strictly on training set
mean = np.mean(X_train, axis=0)
std = np.std(X_train, axis=0) + 1e-7
X_train_std = (X_train - mean) / std
X_test_std = (X_test - mean) / std

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
print("Configured 5-Fold StratifiedGroupKFold on training set.")


In [ ]:
# Cell 4: Train models and collect out-of-fold validation predictions
from src.models.train_and_evaluate import train_and_evaluate_all_models

# Run the complete multi-architecture evaluation pipeline
results = train_and_evaluate_all_models(sample_limit=1000)
df_bench = pd.DataFrame(results["results"])

print("=" * 100)
print("  MULTI-ARCHITECTURE BENCHMARK EVALUATION RESULTS")
print("=" * 100)
print(df_bench[["Model Configuration", "Strategy", "Test Loss", "Accuracy", "Precision", "Recall", "F1-Score"]].to_string(index=False))
print("=" * 100)
